In [10]:
import tkinter as tk
from tkinter import ttk
import time
import random

BANGS = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
MAU_SAC = ["Red", "Green", "Blue"]
MAU_HEX = {"Red": "#ff4d4d", "Green": "#2ecc71", "Blue": "#3498db", None: "#f2f2f2"}

class GiaoDienToMauAC3Chuan:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("Map Coloring - AC3 + Backtracking (MAC Solver)")

        self.mien_gia_tri = {bang: list(MAU_SAC) for bang in BANGS}
        self.phan_bo_mau = {bang: None for bang in BANGS}
        
        self.so_buoc = 0
        self.dang_chay = False
        self.cac_vung_giao_dien = {}
        self.toa_do_hien_tai = {}
        self.giap_ranh_dong = {} 

        khung_tren = tk.Frame(cua_so, bg="#34495e", padx=5, pady=5)
        khung_tren.pack(pady=5, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren, 
            text="Trạng thái: Sẵn sàng | Bước: 0", 
            anchor="w", font=("Arial", 11, "bold"), fg="white", bg="#34495e"
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        self.btn_chay = tk.Button(khung_tren, text="Chạy AC3 + Tô Màu", command=self.bat_dau_chuong_trinh, bg="#27ae60", fg="white", font=("Arial", 10, "bold"))
        self.btn_chay.pack(side="right", padx=5)
        
        tk.Button(khung_tren, text="Đặt Lại Map", command=self.dat_lai_va_sinh_dia_hinh).pack(side="right", padx=5)

        # --- GIAO DIỆN CHÍNH (CANVAS + LOG) ---
        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)

        # Trái: Bản đồ Canvas
        khung_trai = tk.Frame(khung_chinh)
        khung_trai.pack(side="left", anchor="n")

        self.canvas = tk.Canvas(khung_trai, width=440, height=450, bg="#eef2f5", relief="raised", borderwidth=2)
        self.canvas.pack()

        # Phải: Log lộ trình xử lý
        khung_phai = tk.Frame(khung_chinh, padx=10)
        khung_phai.pack(side="left", fill="both", expand=True)
        tk.Label(khung_phai, text="LỘ TRÌNH LỌC AC3 & QUAY LUI TÔ MÀU", font=("Arial", 10, "bold"), fg="#27ae60").pack(anchor="w", pady=2)
        
        thanh_cuon = tk.Scrollbar(khung_phai)
        thanh_cuon.pack(side="right", fill="y")

        self.o_chu_log = tk.Text(
            khung_phai, width=50, height=25, font=("Courier", 10, "bold"),
            bg="#fdfdfd", yscrollcommand=thanh_cuon.set
        )
        self.o_chu_log.pack(side="left", fill="both", expand=True)
        thanh_cuon.config(command=self.o_chu_log.yview)
        
        self.dat_lai_va_sinh_dia_hinh()

    def sinh_ban_do_so_le_tu_nhien(self):
        w_trai = random.randint(110, 140); w_giua = random.randint(110, 135); w_phai = random.randint(110, 135)
        h_hang1 = random.randint(100, 125); h_hang2 = random.randint(110, 135); h_hang3 = random.randint(75, 95)
        dy_trai = random.randint(10, 35); dy_giua = random.randint(-15, 15); dy_phai = random.randint(15, 40)
        dx_sa = random.randint(-20, 20); dx_nsw = random.randint(10, 30); dx_v = random.randint(-25, 25)
        x_goc, y_goc = 40, 50
        x1_l, y1_l = x_goc, y_goc + dy_trai
        x2_l, y2_l = x1_l + w_trai, y1_l + h_hang1 + h_hang2
        x1_m1, y1_m1 = x2_l, y_goc + dy_giua
        x2_m1, y2_m1 = x1_m1 + w_giua, y1_m1 + h_hang1
        x1_r1, y1_r1 = x2_m1, y_goc + dy_phai
        x2_r1, y2_r1 = x1_r1 + w_phai, y1_r1 + h_hang1 + 20
        x1_m2, y1_m2 = x2_l + dx_sa, y2_m1
        x2_m2, y2_m2 = x2_m1 + 10, y1_m2 + h_hang2
        x1_r2, y1_r2 = x2_m2, y2_r1
        x2_r2, y2_r2 = x2_r1 - dx_nsw, y1_r2 + (y2_m2 - y1_r2) + 15
        x1_b, y1_b = x2_l + dx_v, y2_m2
        x2_b, y2_b = x2_r2, y1_b + h_hang3
        x1_t, y1_t = x1_b + random.randint(30, 70), y2_b + 15
        x2_t, y2_t = x1_t + random.randint(110, 160), y1_t + 45
        vung_dat_phat_sinh = [
            [x1_l, y1_l, x2_l, y1_l, x2_l, y2_l, x1_l, y2_l],
            [x1_m1, y1_m1, x2_m1, y1_m1, x2_m1, y2_m1, x1_m1, y2_m1],
            [x1_r1, y1_r1, x2_r1, y1_r1, x2_r1, y2_r1, x1_r1, y2_r1],
            [x1_m2, y1_m2, x2_m2, y1_m2, x2_m2, y2_m2, x1_m2, y2_m2],
            [x1_r2, y1_r2, x2_r2, y1_r2, x2_r2, y2_r2, x1_r2, y2_r2],
            [x1_b, y1_b, x2_b, y1_b, x2_b, y2_b, x1_b, y2_b],
            [x1_t, y1_t, x2_t, y1_t, x2_t, y2_t, x1_t, y2_t]
        ]
        random.shuffle(vung_dat_phat_sinh)
        self.toa_do_hien_tai = {}
        for idx, bang in enumerate(BANGS):
            self.toa_do_hien_tai[bang] = vung_dat_phat_sinh[idx]
        self.tu_dong_tinh_giap_ranh_hinh_hoc()

    def tu_dong_tinh_giap_ranh_hinh_hoc(self):
        self.giap_ranh_dong = {bang: [] for bang in BANGS}
        sai_so_tiep_xuc = 5 
        for i in range(len(BANGS)):
            for j in range(i + 1, len(BANGS)):
                b1 = BANGS[i]; b2 = BANGS[j]
                box1 = self.toa_do_hien_tai[b1]; box2 = self.toa_do_hien_tai[b2]
                min_x1, min_y1 = box1[0], box1[1]; max_x1, max_y1 = box1[4], box1[5]
                min_x2, min_y2 = box2[0], box2[1]; max_x2, max_y2 = box2[4], box2[5]
                giao_x = not (max_x1 < min_x2 - sai_so_tiep_xuc or min_x1 > max_x2 + sai_so_tiep_xuc)
                giao_y = not (max_y1 < min_y2 - sai_so_tiep_xuc or min_y1 > max_y2 + sai_so_tiep_xuc)
                if giao_x and giao_y:
                    self.giap_ranh_dong[b1].append(b2)
                    self.giap_ranh_dong[b2].append(b1)

    def ve_ban_do(self):
        self.canvas.delete("all")
        self.cac_vung_giao_dien = {}
        for bang in BANGS:
            toa_do = self.toa_do_hien_tai[bang]
            mau_hien_tai = self.phan_bo_mau[bang]
            mau_nen = MAU_HEX[mau_hien_tai]
            
            poly_id = self.canvas.create_polygon(toa_do, fill=mau_nen, outline="#2c3e50", width=2)
            
            xs = toa_do[0::2]; ys = toa_do[1::2]
            cx = sum(xs) / len(xs); cy = sum(ys) / len(ys)
            
            mau_chu = "white" if mau_hien_tai in ["Red", "Blue"] else "black"
            
            chu_mien = "".join([m[0] for m in self.mien_gia_tri[bang]])
            text_hien_thi = f"{bang}\n[{chu_mien}]"
            
            txt_id = self.canvas.create_text(cx, cy, text=text_hien_thi, font=("Arial", 10, "bold"), fill=mau_chu, justify="center")
            self.cac_vung_giao_dien[bang] = (poly_id, txt_id)

    def cap_nhat_thanh_trang_thai(self, noi_dung_chu, mau_nen="#34495e"):
        chu_thong_ke = f"{noi_dung_chu} | Bước: {self.so_buoc}"
        self.nhan_trang_thai.config(text=chu_thong_ke, bg=mau_nen)
        self.nhan_trang_thai.master.config(bg=mau_nen)

    def conflict(self, bang_dang_xet, mau_du_dinh):
        for hang_xom in self.giap_ranh_dong[bang_dang_xet]:
            if self.phan_bo_mau[hang_xom] == mau_du_dinh: 
                return True 
        return False

    def AC3_Giai_Doan_Dau(self):
        self._ghi_mot_dong_log("🔄 BƯỚC 1: Kích hoạt bộ lọc AC3 để tỉa cung lỗi...")
        hang_doi_cung = []
        for Xi in BANGS:
            for Xj in self.giap_ranh_dong[Xi]:
                hang_doi_cung.append((Xi, Xj))
                
        while hang_doi_cung:
            Xi, Xj = hang_doi_cung.pop(0)
            if self.Remove_Inconsistent_Values(Xi, Xj):
                if len(self.mien_gia_tri[Xi]) == 0:
                    return False
                for Xk in self.giap_ranh_dong[Xi]:
                    if Xk != Xj:
                        hang_doi_cung.append((Xk, Xi))
                        
        self._ghi_mot_dong_log("✅ AC3 lọc xong! Miền giá trị đồ thị đạt chuẩn tối ưu sạch sẽ.")
        self.ve_ban_co_va_update()
        return True

    def Remove_Inconsistent_Values(self, Xi, Xj):
        bi_loai_bo = False
        for mau_x in list(self.mien_gia_tri[Xi]):
            hop_le = False
            for mau_y in self.mien_gia_tri[Xj]:
                if mau_x != mau_y:
                    hop_le = True
                    break
            if not hop_le:
                self.mien_gia_tri[Xi].remove(mau_x)
                bi_loai_bo = True
        return bi_loai_bo
    def backtracking_to_mau(self, index_bang):
        if index_bang == len(BANGS):
            return True

        bang_hien_tai = BANGS[index_bang]
        
        for mau in list(self.mien_gia_tri[bang_hien_tai]):
            self.so_buoc += 1
            self.cap_nhat_thanh_trang_thai("Trạng thái: Đang nhuộm màu...", mau_nen="#34495e")
            
            if not self.conflict(bang_hien_tai, mau):
                self.phan_bo_mau[bang_hien_tai] = mau
                self._ghi_mot_dong_log(f"Bước {self.so_buoc:02d}: Tô màu [{mau}] cho vùng [{bang_hien_tai}]")
                self.ve_ban_co_va_update()
                time.sleep(0.4)
                
                if self.backtracking_to_mau(index_bang + 1):
                    return True
                
                self.phan_bo_mau[bang_hien_tai] = None
                self._ghi_mot_dong_log(f"  <- Thất bại! Quay lui xóa màu tại [{bang_hien_tai}]")
                self.ve_ban_co_va_update()
                time.sleep(0.2)
                
        return False

    def bat_dau_chuong_trinh(self):
        if self.dang_chay: return
        self.dang_chay = True
        self.btn_chay.config(state="disabled")
        
        self.o_chu_log.config(state="normal")
        self.o_chu_log.delete(1.0, tk.END)
        self.o_chu_log.config(state="disabled")
        
        self.so_buoc = 0
        
        if self.AC3_Giai_Doan_Dau():
            self._ghi_mot_dong_log("\n🎨 BƯỚC 2: Khởi động hệ thống nhuộm màu Backtracking...")
            thanh_cong = self.backtracking_to_mau(0)
            
            if thanh_cong:
                self.cap_nhat_thanh_trang_thai("Trạng thái: THÀNH CÔNG ", mau_nen="#27ae60")
            else:
                self.cap_nhat_thanh_trang_thai("Trạng thái: THẤT BẠI ", mau_nen="#7f8c8d")
        else:
            self.cap_nhat_thanh_trang_thai("Trạng thái: LỖI (AC3 PHÁT HIỆN SỚM VÔ NGHIỆM)", mau_nen="#7f8c8d")
            
        self.dang_chay = False

    def dat_lai_va_sinh_dia_hinh(self):
        self.dang_chay = False
        self.mien_gia_tri = {bang: list(MAU_SAC) for bang in BANGS}
        self.phan_bo_mau = {bang: None for bang in BANGS}
        self.so_buoc = 0
        
        self.sinh_ban_do_so_le_tu_nhien()
        self.ve_ban_do()
        
        self.o_chu_log.config(state="normal")
        self.o_chu_log.delete(1.0, tk.END)
        self.o_chu_log.config(state="disabled")
        
        self.btn_chay.config(state="normal")
        self.cap_nhat_thanh_trang_thai("Trạng thái: SẠCH SẼ - ĐÃ ĐẶT LẠI MAP", mau_nen="#34495e")

    def ve_ban_co_va_update(self):
        self.ve_ban_do()
        self.cua_so.update()

    def _ghi_mot_dong_log(self, txt):
        self.o_chu_log.config(state="normal")
        self.o_chu_log.insert(tk.END, f"{txt}\n")
        self.o_chu_log.see(tk.END)
        self.o_chu_log.config(state="disabled")

if __name__ == "__main__":
    root = tk.Tk()
    app = GiaoDienToMauAC3Chuan(root)
    root.mainloop()